In [ ]:
from pyspark.sql import functions as F

df_sol = spark.sql(
    """
    SELECT * FROM silver.fato_solicitacoes 
    WHERE municipio = 'Osasco'
    """
)

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.

In [1]:
from pyspark.sql import functions as F

df_sol = spark.sql(
    """
    SELECT * FROM silver.fato_solicitacoes 
    WHERE municipio = 'Osasco' 
    AND servico = 'Atendimento ao trabalhador'
    """
)

df_campos = spark.sql(
    """
    SELECT * FROM silver.fato_campos 
    WHERE municipio = 'Osasco' 
    AND servico = 'Atendimento ao trabalhador'
    """
)

df_campos_pivot = (
    df_campos
    .groupBy("id_os")
    .pivot("campo")
    .agg(F.first("valor"))
)

df = df_sol.join(df_campos_pivot, on="id_os", how="left")

colunas_demanda = [c for c in df.columns if c.startswith("demanda_")]
for c in colunas_demanda:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", None).otherwise(F.col(c))
        .cast("int")
    )
df = df.fillna(0, subset=colunas_demanda)

df = df.withColumn(
    "tempo_atendimento_minutos",
    (F.col("data_finalizacao").cast("long") - F.col("data_criacao").cast("long")) / 60
)

StatementMeta(, 8df23e25-68fa-41dc-843b-4bc4d9a82c02, 3, Finished, Available, Finished, False)

In [6]:
(
    df
    .write.mode("overwrite")
    .format("delta")
    .saveAsTable("gold.osasco_atendimento_trabalhador")
)

StatementMeta(, 8df23e25-68fa-41dc-843b-4bc4d9a82c02, 8, Finished, Available, Finished, False)